# 动态系统提示
对于需要根据运行时上下文或智能体状态修改系统提示的高级用例
@dynamic_prompt 装饰器创建中间件，根据模型请求动态生成系统提示：


In [ ]:
import os
from typing import TypedDict
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.chat_models import init_chat_model


class Context(TypedDict):
    """
    给字典起个名字 + 定规矩
        不是类！
        不是对象！
        不是实例化！
        它 = 带类型约束的字典

    TypedDict 只在「写代码时」检查：
        少传字段 → 报错
        类型错 → 报错
        运行时 → 还是字典
        不创建对象
        不占额外内存
    
    为什么要用它？（工程上为什么这么写）
        因为在 LangGraph / Agent 里：
        我们要传递上下文（context）
        上下文本质就是字典
        但字典太自由，容易写错字段名
        所以用 TypedDict 约束格式，更安全、更清晰
    """
    user_role: str


@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """根据用户角色生成系统提示。"""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "你是一个有帮助的助手。"

    if user_role == "expert":
        return f"{base_prompt} 提供详细的技术响应。"
    elif user_role == "beginner":
        return f"{base_prompt} 简单解释概念，避免使用行话。"

    return base_prompt


model = init_chat_model(
    model="qwen-plus",  # 模型名称可自定义
    model_provider="openai",  # 如果是Langchain不支持的模型类型，需要指定模型提供者，虽然Langchain不支持阿里千问，但是千问兼容openai
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

agent = create_agent(
    model=model,
    # tools=[web_search],
    middleware=[user_role_prompt],
    context_schema=Context,
)

# 系统提示将根据上下文动态设置
result = agent.invoke(
    {"messages": [{"role": "user", "content": "解释机器学习"}]},
    context={"user_role": "expert"},
)